<a href="https://colab.research.google.com/github/haris444/autonomous-agents/blob/dev/train_strong_social_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Strong Social Rewards — Notebook 1/2

**H: Strong Social** (1 predator) + **H-NoPred: Strong Social, No Predators**

Tests whether strong ledger-based social rewards (reciprocity=3, betrayal=-5, defense=10, food_share=3, revenge=5) improve performance over the D baseline (social rewards = 0).

| Run | Predators | Compare to | Key question |
|---|---|---|---|
| H: strong_social | 1 | D (+265) | Do social instincts help in standard env? |
| H-NoPred: strong_social_no_pred | 0 | C (+689) | Do social instincts help without predators? |

**Runtime:** ~4-5 hours on T4 GPU (2 runs x 10M steps, 4 envs each, sequential)

# Strong Social Rewards — Notebook 1/2

**H: Strong Social** (1 predator) + **H-NoPred: Strong Social, No Predators**

Tests whether strong ledger-based social rewards (reciprocity=3, betrayal=-5, defense=10, food_share=3, revenge=5) improve performance over the D baseline (social rewards = 0).

| Run | Predators | Compare to | Key question |
|---|---|---|---|
| H: strong_social | 1 | D (+265) | Do social instincts help in standard env? |
| H-NoPred: strong_social_no_pred | 0 | C (+689) | Do social instincts help without predators? |

**Runtime:** ~2-3 hours on T4 GPU (2 runs x 5M steps, 4 envs each, sequential)

In [ ]:
!git clone -b dev https://github.com/haris444/autonomous-agents.git
%cd autonomous-agents
!pip install -q pyyaml

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Run H: Strong Social (1 predator)

In [ ]:
!mkdir -p sac_h_strong_social
!python -m training.train_sac \
    --experiment experiments/configs/sac_h_strong_social.yaml \
    --csv-log sac_h_strong_social/log.csv \
    --output-dir sac_h_strong_social \
    --ckpt-every 500

## 3. Run H-NoPred: Strong Social (no predators)

In [ ]:
!mkdir -p sac_h_strong_social_no_pred
!python -m training.train_sac \
    --experiment experiments/configs/sac_h_strong_social_no_pred.yaml \
    --csv-log sac_h_strong_social_no_pred/log.csv \
    --output-dir sac_h_strong_social_no_pred \
    --ckpt-every 500

## 4. Comparison Plots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

runs = {
    'H: Strong Social (1 pred)': ('sac_h_strong_social/log.csv', '#1f77b4'),
    'H-NoPred: Strong Social (0 pred)': ('sac_h_strong_social_no_pred/log.csv', '#2ca02c'),
}

dfs = {}
for label, (path, _) in runs.items():
    try:
        dfs[label] = pd.read_csv(path)
    except Exception as e:
        print(f'Warning: {label} -- {e}')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Strong Social Rewards — Notebook 1', fontsize=14, fontweight='bold')

# Episode Return
ax = axes[0, 0]
for label, df in dfs.items():
    ax.plot(df['global_step'], df['episode_return'].rolling(50).mean(),
            color=runs[label][1], linewidth=2, label=label)
ax.set_xlabel('Global Step')
ax.set_ylabel('Episode Return')
ax.set_title('Episode Return (smoothed)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Q-values
ax = axes[0, 1]
for label, df in dfs.items():
    q1 = df['q1_mean'].replace(0, float('nan'))
    ax.plot(df['global_step'], q1.rolling(50).mean(),
            color=runs[label][1], linewidth=2, label=label)
ax.set_xlabel('Global Step')
ax.set_ylabel('Q1 Mean')
ax.set_title('Critic Q-Values')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Alpha (direction)
ax = axes[1, 0]
for label, df in dfs.items():
    ad = df['alpha_dir'].replace(0, float('nan'))
    ax.plot(df['global_step'], ad, color=runs[label][1], linewidth=1.5, label=label)
ax.set_xlabel('Global Step')
ax.set_ylabel('Alpha (direction)')
ax.set_title('Entropy Temperature (Direction)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Alpha (action type)
ax = axes[1, 1]
for label, df in dfs.items():
    aa = df['alpha_act'].replace(0, float('nan'))
    ax.plot(df['global_step'], aa, color=runs[label][1], linewidth=1.5, label=label)
ax.set_xlabel('Global Step')
ax.set_ylabel('Alpha (action type)')
ax.set_title('Entropy Temperature (Action Type)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('strong_social_nb1_comparison.png', dpi=150)
plt.show()

print('\nResults:')
print(f'{"Variant":<40} {"Final Return":>14} {"Episodes":>10} {"Final Q1":>10}')
print('-' * 76)
for label, df in dfs.items():
    ret = df['episode_return'].iloc[-1]
    eps = df['episodes_completed'].iloc[-1]
    q1 = df['q1_mean'].iloc[-1]
    print(f'{label:<40} {ret:>14.1f} {eps:>10} {q1:>10.2f}')

## 5. Save to Google Drive

In [ ]:
from google.colab import drive
import shutil, os, glob

drive.mount('/content/drive')

drive_dir = '/content/drive/MyDrive/AutonomousAgents/strong_social'
os.makedirs(drive_dir, exist_ok=True)

# Save comparison plot
if os.path.exists('strong_social_nb1_comparison.png'):
    shutil.copy2('strong_social_nb1_comparison.png', drive_dir)

for run_dir in ['sac_h_strong_social', 'sac_h_strong_social_no_pred']:
    var_drive = f'{drive_dir}/{run_dir}'
    os.makedirs(var_drive, exist_ok=True)

    for f in ['log.csv', 'training_curves.png']:
        p = f'{run_dir}/{f}'
        if os.path.exists(p):
            shutil.copy2(p, var_drive)

    ckpts = sorted(glob.glob(f'{run_dir}/checkpoint_*.pt'))
    for ckpt in ckpts:
        shutil.copy2(ckpt, var_drive)
    n_ckpts = len(ckpts)
    first = os.path.basename(ckpts[0]) if ckpts else 'none'
    last = os.path.basename(ckpts[-1]) if ckpts else 'none'
    print(f'{run_dir}: {n_ckpts} checkpoints ({first} -> {last})')

print(f'\nAll results saved to: {drive_dir}')